# 05: NLP — From Bag-of-Words to Transformers

## Module 2, Week 8: Natural Language Processing

---

### Learning Objectives
By the end of this notebook, you will:
1. **Preprocess text** — tokenization, lowercasing, stopwords
2. **Represent text as numbers** — Bag-of-Words and TF-IDF
3. **Understand word embeddings** — Word2Vec, GloVe, and semantic similarity
4. **Know the evolution** — RNNs → LSTMs → Transformers (conceptually)
5. **Understand the Transformer** — self-attention, encoder/decoder, BERT vs GPT
6. **Use HuggingFace** — pretrained models in 3 lines of code

### Roadmap

```
Classical NLP          Word Embeddings       Sequence Models        Transformers
─────────────────   →  ─────────────────  →  ─────────────────  →  ─────────────────
BoW, TF-IDF             Word2Vec, GloVe       RNN, LSTM, GRU        Attention, BERT,
(sparse, no order)      (dense, semantic)     (sequential, slow)     GPT, HuggingFace
                                                                     (parallel, SOTA)
```

> **The big picture:** NLP has evolved from counting words to understanding meaning. Each step solved a limitation of the previous approach. Today, Transformers dominate — but understanding the journey helps you pick the right tool.

In [ ]:
import sys
sys.path.insert(0, '../../..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

from shared.viz_utils import setup_style, save_fig
setup_style()

print("Setup complete!")

---
## Section 1: Text Preprocessing — Turning Text into Numbers

The fundamental challenge of NLP: **machines understand numbers, not words.**

Before any model can process text, we need to convert it into a numerical representation. The preprocessing pipeline typically looks like:

```
Raw Text  →  Tokenize  →  Lowercase  →  Remove Stopwords  →  Stem/Lemmatize  →  Numbers
```

**Key steps:**
- **Tokenization** — splitting text into individual words (tokens)
- **Lowercasing** — "Movie" and "movie" should be the same word
- **Stopword removal** — "the", "is", "a" carry little meaning
- **Stemming** — reducing words to their root: "running" → "run"
- **Lemmatization** — smarter stemming using grammar: "better" → "good"

Let's start with a small movie review dataset to see this in action.

In [ ]:
# Sample corpus — movie reviews with sentiment labels
reviews = [
    "This movie was absolutely fantastic! Best film I've seen all year.",
    "Terrible waste of time. The acting was horrible and the plot made no sense.",
    "A decent movie with some good moments, but nothing special overall.",
    "Loved every minute of it! Great performances and stunning visuals.",
    "Boring and predictable. I fell asleep halfway through.",
    "An okay film. Some parts were interesting but it dragged on too long.",
    "Masterpiece! This director never disappoints. A must-watch!",
    "Worst movie ever. Don't waste your money on this garbage.",
    "Pretty good! Not perfect, but entertaining and fun.",
    "Disappointing sequel. The original was so much better.",
]
labels = [1, 0, 1, 1, 0, 0, 1, 0, 1, 0]  # 1=positive, 0=negative

# Let's see our data
for i, (review, label) in enumerate(zip(reviews, labels)):
    sentiment = "Positive" if label == 1 else "Negative"
    print(f"[{sentiment:8s}] {review}")

In [ ]:
# Step 1: Tokenization — splitting text into words
sample = reviews[0]
print(f"Original: {sample}")
print()

# Simple tokenization: split on whitespace
tokens_simple = sample.split()
print(f"Simple split: {tokens_simple}")
print(f"Number of tokens: {len(tokens_simple)}")
print()

# Better tokenization: lowercase + remove punctuation
import re
tokens_clean = re.findall(r'\b[a-z]+\b', sample.lower())
print(f"Clean tokens: {tokens_clean}")
print(f"Number of tokens: {len(tokens_clean)}")
print()

# Common English stopwords (a small subset)
stopwords = {'the', 'a', 'an', 'is', 'was', 'this', 'it', 'of', 'and', 'i', 
             'to', 'in', 'that', 'with', 'for', 'on', 'but', 'all', 've', 'no',
             'some', 'not', 'so', 'my', 'we', 'do', 'don'}
tokens_filtered = [t for t in tokens_clean if t not in stopwords]
print(f"After stopword removal: {tokens_filtered}")
print(f"Number of tokens: {len(tokens_filtered)}")
print()

# Word frequency across all reviews
from collections import Counter
all_tokens = []
for review in reviews:
    all_tokens.extend(re.findall(r'\b[a-z]+\b', review.lower()))

word_counts = Counter(all_tokens)
print("Top 15 most common words:")
for word, count in word_counts.most_common(15):
    print(f"  {word:15s} → {count}")

---
## Section 2: Bag of Words (BoW)

The simplest way to turn text into numbers: **count how many times each word appears.**

Each document becomes a vector where:
- Each dimension = one word from the vocabulary
- Each value = count of that word in the document

```
Vocabulary:  [amazing, bad, boring, film, good, great, movie, terrible, ...]
Document 1:  [  0,      0,    0,     1,    0,     0,     1,      0,     ...]
Document 2:  [  0,      1,    1,     0,    0,     0,     1,      1,     ...]
```

**Key properties:**
- Very sparse (mostly zeros — each document uses only a fraction of all words)
- Completely loses word order ("dog bites man" = "man bites dog")
- Simple but surprisingly effective as a baseline!

In [ ]:
# Bag of Words with sklearn's CountVectorizer
bow_vectorizer = CountVectorizer()
bow_matrix = bow_vectorizer.fit_transform(reviews)

# What's in our vocabulary?
vocab = bow_vectorizer.get_feature_names_out()
print(f"Vocabulary size: {len(vocab)} unique words")
print(f"Vocabulary: {list(vocab)}")
print()

# Shape of the matrix
print(f"BoW matrix shape: {bow_matrix.shape}")
print(f"  → {bow_matrix.shape[0]} documents × {bow_matrix.shape[1]} words")
print(f"  → {bow_matrix.nnz} non-zero entries out of {bow_matrix.shape[0] * bow_matrix.shape[1]} total")
print(f"  → {100 * bow_matrix.nnz / (bow_matrix.shape[0] * bow_matrix.shape[1]):.1f}% non-zero (very sparse!)")
print()

# Let's look at the first review as a BoW vector
print(f"Review 1: '{reviews[0]}'")
print(f"BoW vector (non-zero entries):")
row = bow_matrix[0].toarray().flatten()
for word, count in zip(vocab, row):
    if count > 0:
        print(f"  {word}: {count}")

In [ ]:
# Visualize: top words in positive vs negative reviews
bow_array = bow_matrix.toarray()
labels_arr = np.array(labels)

pos_word_freq = bow_array[labels_arr == 1].sum(axis=0)
neg_word_freq = bow_array[labels_arr == 0].sum(axis=0)

# Create a comparison DataFrame
word_df = pd.DataFrame({
    'word': vocab,
    'positive': pos_word_freq,
    'negative': neg_word_freq
})
word_df['difference'] = word_df['positive'] - word_df['negative']
word_df = word_df.sort_values('difference')

# Plot the most distinguishing words
fig, ax = plt.subplots(figsize=(10, 6))
top_n = 12
plot_df = pd.concat([word_df.head(top_n // 2), word_df.tail(top_n // 2)])
colors = ['#e74c3c' if d < 0 else '#2ecc71' for d in plot_df['difference']]
ax.barh(plot_df['word'], plot_df['difference'], color=colors)
ax.set_xlabel('Word Count Difference (Positive - Negative)')
ax.set_title('Most Distinguishing Words: Bag of Words')
ax.axvline(x=0, color='black', linewidth=0.5)
ax.annotate('← More Negative', xy=(0.15, 0.02), xycoords='axes fraction', color='#e74c3c', fontsize=10)
ax.annotate('More Positive →', xy=(0.7, 0.02), xycoords='axes fraction', color='#2ecc71', fontsize=10)
plt.tight_layout()
save_fig(fig, '05_bow_distinguishing_words')
plt.show()

---
## Section 3: TF-IDF — Smarter Word Weighting

**Problem with BoW:** common words like "the", "movie", "was" get high counts but carry little meaning.

**TF-IDF** (Term Frequency - Inverse Document Frequency) fixes this:

$$\text{TF-IDF}(word, doc) = \underbrace{\text{TF}(word, doc)}_{\text{How often in this doc?}} \times \underbrace{\text{IDF}(word)}_{\text{How rare across all docs?}}$$

- **TF** (Term Frequency): How often does the word appear in *this* document?
- **IDF** (Inverse Document Frequency): How rare is the word across *all* documents?
  - IDF = log(total docs / docs containing word)
  - "the" appears in every doc → IDF ≈ 0 (not informative)
  - "masterpiece" appears in 1 doc → IDF is high (very informative)

**Result:** Words that are frequent in one document but rare overall get the highest scores — exactly the words that distinguish documents from each other.

In [ ]:
# TF-IDF with sklearn
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(reviews)

# Compare BoW vs TF-IDF for the same document
doc_idx = 1  # "Terrible waste of time..."
print(f"Review: '{reviews[doc_idx]}'")
print()

bow_row = bow_matrix[doc_idx].toarray().flatten()
tfidf_row = tfidf_matrix[doc_idx].toarray().flatten()

# Show non-zero entries side by side
comparison = []
for word, bow_val, tfidf_val in zip(vocab, bow_row, tfidf_row):
    if bow_val > 0 or tfidf_val > 0:
        comparison.append({'word': word, 'BoW (count)': int(bow_val), 'TF-IDF (weighted)': round(tfidf_val, 3)})

comp_df = pd.DataFrame(comparison).sort_values('TF-IDF (weighted)', ascending=False)
print("BoW vs TF-IDF comparison:")
print(comp_df.to_string(index=False))
print()

# Key insight: common words get downweighted
print("Key insight:")
print("  'the' appears in many reviews → low TF-IDF score")
print("  'terrible', 'horrible' are distinctive → high TF-IDF scores")

In [ ]:
# Visualize: most distinctive TF-IDF words per class
tfidf_array = tfidf_matrix.toarray()

pos_tfidf = tfidf_array[labels_arr == 1].mean(axis=0)
neg_tfidf = tfidf_array[labels_arr == 0].mean(axis=0)

tfidf_diff = pd.DataFrame({
    'word': vocab,
    'positive_avg': pos_tfidf,
    'negative_avg': neg_tfidf,
    'difference': pos_tfidf - neg_tfidf
}).sort_values('difference')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Top negative words
top_neg = tfidf_diff.head(8)
axes[0].barh(top_neg['word'], top_neg['negative_avg'], color='#e74c3c')
axes[0].set_title('Top TF-IDF Words in Negative Reviews')
axes[0].set_xlabel('Average TF-IDF Score')

# Top positive words
top_pos = tfidf_diff.tail(8)
axes[1].barh(top_pos['word'], top_pos['positive_avg'], color='#2ecc71')
axes[1].set_title('Top TF-IDF Words in Positive Reviews')
axes[1].set_xlabel('Average TF-IDF Score')

plt.tight_layout()
save_fig(fig, '05_tfidf_distinctive_words')
plt.show()

---
## Section 4: Text Classification with sklearn

Let's build a proper text classification pipeline using a larger dataset. We'll use sklearn's **20 Newsgroups** dataset — a classic benchmark of ~20,000 newsgroup posts across 20 topics.

**Pipeline:** Raw Text → TF-IDF Vectorizer → Logistic Regression → Prediction

This is the standard approach for classical NLP, and it's still a great baseline!

In [ ]:
from sklearn.datasets import fetch_20newsgroups

# Pick 4 distinct categories for a cleaner demo
categories = ['sci.space', 'rec.sport.baseball', 'comp.graphics', 'talk.politics.mideast']

# Load train and test sets
newsgroups_train = fetch_20newsgroups(subset='train', categories=categories, 
                                      remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', categories=categories,
                                     remove=('headers', 'footers', 'quotes'))

print(f"Training samples: {len(newsgroups_train.data)}")
print(f"Test samples: {len(newsgroups_test.data)}")
print(f"Categories: {newsgroups_train.target_names}")
print()

# Show an example
print("--- Example document (sci.space) ---")
space_idx = np.where(newsgroups_train.target == newsgroups_train.target_names.index('sci.space'))[0][0]
print(newsgroups_train.data[space_idx][:300] + "...")

In [ ]:
# Build a Pipeline: TF-IDF → Logistic Regression
text_clf = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=10000, stop_words='english')),
    ('clf', LogisticRegression(max_iter=1000, C=1.0))
])

# Train the pipeline
text_clf.fit(newsgroups_train.data, newsgroups_train.target)

# Evaluate on test set
y_pred = text_clf.predict(newsgroups_test.data)
print("Classification Report:")
print("=" * 65)
print(classification_report(newsgroups_test.target, y_pred, 
                            target_names=newsgroups_train.target_names))

# Cross-validation on training set
cv_scores = cross_val_score(text_clf, newsgroups_train.data, newsgroups_train.target, cv=5)
print(f"Cross-validation accuracy: {cv_scores.mean():.3f} (+/- {cv_scores.std():.3f})")
print()
print("Not bad for such a simple approach!")

In [ ]:
# Most informative features — what words define each category?
feature_names = text_clf.named_steps['tfidf'].get_feature_names_out()
coefs = text_clf.named_steps['clf'].coef_

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, (ax, category) in enumerate(zip(axes.flat, newsgroups_train.target_names)):
    top_n = 10
    top_positive = np.argsort(coefs[idx])[-top_n:]
    
    words = [feature_names[i] for i in top_positive]
    weights = [coefs[idx][i] for i in top_positive]
    
    ax.barh(words, weights, color=plt.cm.Set2(idx))
    ax.set_title(f'Top words for: {category}')
    ax.set_xlabel('Coefficient Weight')

plt.suptitle('Most Informative Features per Category (TF-IDF + LogReg)', fontsize=14, y=1.02)
plt.tight_layout()
save_fig(fig, '05_informative_features')
plt.show()

---
## Section 5: Word Embeddings — Words as Vectors

### The Limitation of BoW/TF-IDF

BoW and TF-IDF treat every word as independent. They have no concept of **semantic similarity**:
- "good" and "great" are as different as "good" and "terrible"
- "cat" and "kitten" share no relationship

### The Embedding Solution

**Word embeddings** (Word2Vec, GloVe, FastText) learn to represent words as **dense vectors** in a continuous space, where:
- Similar words are close together
- Relationships are captured as directions

The famous example:

```
king - man + woman ≈ queen
```

This works because the vector from "man" to "king" captures the concept of "royalty", and adding that direction to "woman" gives you "queen".

```
        man ──────→ king
         |           |
    (gender)    (gender)
         |           |
        woman ─────→ queen
```

### How are embeddings learned?

**Word2Vec** (2013) — Two approaches:
- **CBOW** (Continuous Bag of Words): predict the center word from context words
- **Skip-gram**: predict context words from the center word

The key insight: words that appear in similar contexts get similar vectors.

In [ ]:
# Demonstrating the concept of word embeddings with cosine similarity
# We'll create simple embeddings to illustrate the idea

from sklearn.metrics.pairwise import cosine_similarity

# Simulated word embeddings (in practice, these are learned from large corpora)
# Each word is a vector — similar words have similar vectors
np.random.seed(42)

# Create embeddings that capture semantic relationships
# We'll manually craft these to demonstrate the concept
word_embeddings = {
    # Positive sentiment words (similar direction)
    'great':      np.array([0.8, 0.7, 0.1, -0.1]),
    'fantastic':  np.array([0.9, 0.8, 0.2, -0.2]),
    'wonderful':  np.array([0.85, 0.75, 0.15, -0.15]),
    'good':       np.array([0.6, 0.5, 0.1, -0.1]),
    # Negative sentiment words (opposite direction)
    'terrible':   np.array([-0.8, -0.7, 0.1, 0.2]),
    'horrible':   np.array([-0.85, -0.75, 0.15, 0.25]),
    'awful':      np.array([-0.7, -0.6, 0.2, 0.1]),
    'bad':        np.array([-0.5, -0.4, 0.1, 0.15]),
    # Neutral words (orthogonal)
    'movie':      np.array([0.1, 0.0, 0.9, 0.3]),
    'film':       np.array([0.15, 0.05, 0.85, 0.35]),
    'book':       np.array([0.1, -0.05, 0.7, 0.6]),
}

# Compute cosine similarity between all pairs
words = list(word_embeddings.keys())
vectors = np.array([word_embeddings[w] for w in words])
sim_matrix = cosine_similarity(vectors)

# Visualize the similarity matrix
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(sim_matrix, xticklabels=words, yticklabels=words, 
            annot=True, fmt='.2f', cmap='RdYlGn', center=0, 
            vmin=-1, vmax=1, ax=ax)
ax.set_title('Cosine Similarity Between Word Embeddings\n(Simulated to illustrate the concept)')
plt.tight_layout()
save_fig(fig, '05_embedding_similarity')
plt.show()

print("\nKey observations:")
print("  'great' and 'fantastic' have high similarity (both positive)")
print("  'great' and 'terrible' have negative similarity (opposite sentiments)")  
print("  'movie' and 'film' are similar (same concept, different words)")
print("  'movie' and 'great' have low similarity (different semantic dimensions)")

In [ ]:
# Visualize embeddings in 2D using PCA
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
vectors_2d = pca.fit_transform(vectors)

fig, ax = plt.subplots(figsize=(10, 7))

# Color by sentiment category
colors = {'positive': '#2ecc71', 'negative': '#e74c3c', 'neutral': '#3498db'}
categories = {
    'great': 'positive', 'fantastic': 'positive', 'wonderful': 'positive', 'good': 'positive',
    'terrible': 'negative', 'horrible': 'negative', 'awful': 'negative', 'bad': 'negative',
    'movie': 'neutral', 'film': 'neutral', 'book': 'neutral'
}

for i, word in enumerate(words):
    cat = categories[word]
    ax.scatter(vectors_2d[i, 0], vectors_2d[i, 1], c=colors[cat], s=100, zorder=5)
    ax.annotate(word, (vectors_2d[i, 0] + 0.02, vectors_2d[i, 1] + 0.02), fontsize=12, fontweight='bold')

# Add legend
for cat, color in colors.items():
    ax.scatter([], [], c=color, s=100, label=cat.capitalize())
ax.legend(fontsize=11)

ax.set_title('Word Embeddings in 2D (PCA Projection)\nSimilar words cluster together', fontsize=13)
ax.set_xlabel('First Principal Component')
ax.set_ylabel('Second Principal Component')
ax.grid(True, alpha=0.3)
plt.tight_layout()
save_fig(fig, '05_embeddings_2d')
plt.show()

---
## Section 6: Sequence Models — RNNs and LSTMs (Conceptual)

### The Limitation of Embeddings Alone

Word embeddings are great for individual words, but how do we handle **sequences**? The meaning of a sentence depends on word **order**:
- "The movie was not good" vs "The movie was good, not bad"

We need models that process text **sequentially** and maintain context.

### Recurrent Neural Networks (RNNs)

RNNs process text one word at a time, passing a **hidden state** from step to step:

```
    x₁         x₂         x₃         x₄
    │           │           │           │
    ▼           ▼           ▼           ▼
┌───────┐   ┌───────┐   ┌───────┐   ┌───────┐
│  RNN  │──→│  RNN  │──→│  RNN  │──→│  RNN  │──→ output
│  Cell │   │  Cell │   │  Cell │   │  Cell │
└───────┘   └───────┘   └───────┘   └───────┘
   h₀──────→  h₁──────→  h₂──────→  h₃──────→ h₄
   (hidden state carries information forward)
```

**Problem: Vanishing Gradients.** In long sequences, the gradient signal fades during backpropagation. The model "forgets" the beginning of the sentence by the time it reaches the end.

### LSTMs (Long Short-Term Memory)

LSTMs solve this with a **gating mechanism** — three gates that control information flow:

```
┌─────────────────────────────┐
│         LSTM Cell           │
│                             │
│  ┌──────┐  ┌──────┐  ┌──────┐  │
│  │Forget│  │Input │  │Output│  │
│  │ Gate │  │ Gate │  │ Gate │  │
│  └──┬───┘  └──┬───┘  └──┬───┘  │
│     │         │         │      │
│     ▼         ▼         ▼      │
│  What to   What new   What to  │
│  forget    info to add output  │
└─────────────────────────────┘
```

- **Forget gate**: What old information to discard
- **Input gate**: What new information to store
- **Output gate**: What to pass to the next step

**GRUs** (Gated Recurrent Units) are a simplified version with just 2 gates — often work just as well.

> **Note:** We won't implement RNNs/LSTMs here — **Transformers have largely replaced them** for most NLP tasks. But understanding this progression is important: BoW → Embeddings → RNNs → LSTMs → **Transformers**.

---
## Section 7: The Transformer — "Attention Is All You Need"

The **Transformer** (Vaswani et al., 2017) revolutionized NLP. The key insight:

> **Self-attention** — every word can attend to every other word in parallel, regardless of distance.

### Why is this better than RNNs?

| Feature | RNN/LSTM | Transformer |
|---------|----------|-------------|
| Processing | Sequential (word by word) | Parallel (all words at once) |
| Long-range dependencies | Struggles (vanishing gradients) | Handles easily (direct attention) |
| Training speed | Slow (can't parallelize) | Fast (GPU-friendly) |
| Context window | Limited by hidden state size | Full sequence visible |

### The Self-Attention Mechanism

For each word, self-attention asks: **"How much should I pay attention to every other word?"**

```
Input:  "The  cat  sat  on  the  mat"

Attention for "sat":
  The  → 0.05  (low attention)
  cat  → 0.40  (high — who sat?)
  sat  → 0.15  (moderate — self)
  on   → 0.25  (high — where?)
  the  → 0.05  (low attention)
  mat  → 0.10  (moderate — on what?)
```

Computed using three learned projections: **Query (Q)**, **Key (K)**, **Value (V)**

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

### Transformer Architecture Overview

```
┌─────────────────────────────────────────┐
│              TRANSFORMER                 │
│                                          │
│   ENCODER (×N)          DECODER (×N)     │
│   ┌──────────┐         ┌──────────┐     │
│   │Feed Fwd  │         │Feed Fwd  │     │
│   ├──────────┤         ├──────────┤     │
│   │Self-Attn │         │Cross-Attn│←──┐ │
│   ├──────────┤         ├──────────┤   │ │
│   │Pos Embed │         │Mask Attn │   │ │
│   ├──────────┤         ├──────────┤   │ │
│   │Input Emb │         │Output Emb│   │ │
│   └──────────┘         └──────────┘   │ │
│        │                               │ │
│        └───────────────────────────────┘ │
└──────────────────────────────────────────┘
```

### The Big Three: BERT, GPT, and T5

| Model | Architecture | Pre-training Task | Best For | Example Uses |
|-------|-------------|-------------------|----------|-------------|
| **BERT** | Encoder only | Masked language model (fill in blanks) | Understanding | Classification, NER, QA |
| **GPT** | Decoder only | Next token prediction (complete text) | Generation | Text completion, chat, code |
| **T5** | Encoder-Decoder | Text-to-text (every task as text in/out) | Both | Summarization, translation |

**Key insight:** You almost never train these from scratch. Instead, you use **pretrained models** and fine-tune them on your specific task. This is called **transfer learning** — the same idea as using ImageNet-pretrained CNNs for vision tasks.

In [ ]:
# Visualize self-attention: a simplified example
# Let's show what attention weights might look like for a sentence

sentence = ["The", "cat", "sat", "on", "the", "mat"]
n = len(sentence)

# Simulated attention weights (what "sat" attends to)
# In a real transformer, these are learned
attention_matrix = np.array([
    [0.30, 0.10, 0.10, 0.10, 0.30, 0.10],  # The
    [0.05, 0.30, 0.30, 0.10, 0.05, 0.20],  # cat
    [0.05, 0.40, 0.15, 0.25, 0.05, 0.10],  # sat
    [0.05, 0.10, 0.30, 0.15, 0.05, 0.35],  # on
    [0.30, 0.10, 0.10, 0.10, 0.30, 0.10],  # the
    [0.05, 0.15, 0.15, 0.35, 0.05, 0.25],  # mat
])

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(attention_matrix, xticklabels=sentence, yticklabels=sentence,
            annot=True, fmt='.2f', cmap='YlOrRd', ax=ax, cbar_kws={'label': 'Attention Weight'})
ax.set_xlabel('Key (attending TO)', fontsize=12)
ax.set_ylabel('Query (attending FROM)', fontsize=12)
ax.set_title('Self-Attention Weights (Simulated)\n"How much does each word attend to every other word?"', fontsize=13)
plt.tight_layout()
save_fig(fig, '05_self_attention')
plt.show()

print("Notice:")
print("  'sat' pays most attention to 'cat' (who sat?) and 'on' (sat where?)")
print("  'on' pays most attention to 'mat' (on what?) and 'sat' (what's on?)")
print("  'The' attends to itself and the other 'the' (article pattern)")

---
## Section 8: HuggingFace Transformers — Pretrained Models in 3 Lines

[HuggingFace](https://huggingface.co/) makes it incredibly easy to use state-of-the-art NLP models. Their `transformers` library provides:

- **`pipeline()`** — the simplest API. One line for sentiment analysis, NER, summarization, etc.
- **`AutoTokenizer`** — handles text → token IDs for any model
- **`AutoModel`** — loads any pretrained model architecture

```python
# This is all you need for sentiment analysis:
from transformers import pipeline
classifier = pipeline("sentiment-analysis")
result = classifier("I love this movie!")
# → [{'label': 'POSITIVE', 'score': 0.9998}]
```

Let's try it on our movie reviews!

In [ ]:
try:
    from transformers import pipeline, AutoTokenizer
    HF_AVAILABLE = True
    
    # Sentiment Analysis — the "hello world" of HuggingFace
    print("Loading sentiment analysis pipeline...")
    classifier = pipeline("sentiment-analysis")
    
    print("\nSentiment Analysis on Movie Reviews:")
    print("=" * 75)
    results = classifier(reviews)
    
    for review, result, true_label in zip(reviews, results, labels):
        predicted = "Pos" if result['label'] == 'POSITIVE' else "Neg"
        true = "Pos" if true_label == 1 else "Neg"
        match = "✓" if (predicted == "Pos") == (true_label == 1) else "✗"
        print(f"  {match} [{predicted} {result['score']:.2f}] (true: {true}) {review[:55]}...")
    
    # Accuracy
    correct = sum(
        1 for r, l in zip(results, labels)
        if (r['label'] == 'POSITIVE') == (l == 1)
    )
    print(f"\nAccuracy: {correct}/{len(labels)} = {correct/len(labels):.0%}")

except ImportError:
    HF_AVAILABLE = False
    print("HuggingFace transformers not installed.")
    print("Install with: pip install transformers torch")
    print("\nSkipping HuggingFace demos — the concepts still apply!")

In [ ]:
# More HuggingFace Pipelines — showing the versatility

if HF_AVAILABLE:
    # --- Zero-Shot Classification ---
    # Classify text into categories the model was NEVER trained on!
    print("=" * 75)
    print("ZERO-SHOT CLASSIFICATION")
    print("(No training needed — the model generalizes from its pretraining)")
    print("=" * 75)
    
    try:
        zero_shot = pipeline("zero-shot-classification")
        
        test_texts = [
            "The stock market crashed today, losing 500 points.",
            "The new iPhone features an improved camera and longer battery life.",
            "Scientists discovered a new species of deep-sea fish.",
        ]
        candidate_labels = ["finance", "technology", "science", "politics"]
        
        for text in test_texts:
            result = zero_shot(text, candidate_labels)
            top_label = result['labels'][0]
            top_score = result['scores'][0]
            print(f"\n  Text: '{text[:60]}...'")
            print(f"  Predicted: {top_label} ({top_score:.2f})")
            for label, score in zip(result['labels'], result['scores']):
                bar = '█' * int(score * 30)
                print(f"    {label:12s} {score:.3f} {bar}")
    except Exception as e:
        print(f"  Zero-shot classification failed: {e}")
    
    # --- Named Entity Recognition ---
    print("\n" + "=" * 75)
    print("NAMED ENTITY RECOGNITION (NER)")
    print("(Automatically finds people, organizations, locations, etc.)")
    print("=" * 75)
    
    try:
        ner = pipeline("ner", aggregation_strategy="simple")
        
        ner_text = "Elon Musk announced that Tesla will open a new factory in Berlin next year."
        entities = ner(ner_text)
        
        print(f"\n  Text: '{ner_text}'")
        print(f"\n  Entities found:")
        for entity in entities:
            print(f"    [{entity['entity_group']:6s}] '{entity['word']}' (score: {entity['score']:.2f})")
    except Exception as e:
        print(f"  NER failed: {e}")

    # --- Summarization ---
    print("\n" + "=" * 75)
    print("TEXT SUMMARIZATION")
    print("(Condense long text into key points)")
    print("=" * 75)
    
    try:
        summarizer = pipeline("summarization", model="sshleifer/distilbart-cnn-12-6")
        
        long_text = """
        The Transformer architecture, introduced in the paper "Attention Is All You Need" 
        by Vaswani et al. in 2017, has fundamentally changed natural language processing. 
        Unlike recurrent neural networks that process sequences one token at a time, 
        Transformers use a self-attention mechanism that allows them to process all tokens 
        in parallel. This not only makes training much faster on modern GPU hardware, 
        but also allows the model to capture long-range dependencies more effectively. 
        The architecture consists of an encoder and decoder, each made up of multiple 
        layers of self-attention and feed-forward networks. Since its introduction, 
        the Transformer has been the foundation for models like BERT, GPT, and T5, 
        which have set new state-of-the-art results on virtually every NLP benchmark.
        """
        
        summary = summarizer(long_text, max_length=50, min_length=20, do_sample=False)
        print(f"\n  Original ({len(long_text.split())} words):")
        print(f"  {long_text.strip()[:150]}...")
        print(f"\n  Summary ({len(summary[0]['summary_text'].split())} words):")
        print(f"  {summary[0]['summary_text']}")
    except Exception as e:
        print(f"  Summarization failed: {e}")

else:
    print("Skipping HuggingFace pipeline demos (transformers not installed)")
    print()
    print("Available pipelines include:")
    print("  - sentiment-analysis: Positive/Negative classification")
    print("  - zero-shot-classification: Classify into any categories without training")
    print("  - ner: Find people, organizations, locations in text")
    print("  - summarization: Condense long documents")
    print("  - translation: Translate between languages")
    print("  - text-generation: Complete/generate text")
    print("  - question-answering: Answer questions given a context paragraph")

---
## Section 9: Tokenization Deep Dive — How Transformers See Text

Transformers don't see words the way we do. They use **subword tokenization** — breaking words into smaller pieces.

### Why Subwords?

- **Word-level** tokenization creates huge vocabularies and can't handle unknown words
- **Character-level** tokenization loses meaning ("c-a-t" doesn't carry "cat" meaning)
- **Subword** tokenization is the sweet spot: common words stay whole, rare words get split

### BERT uses WordPiece tokenization:
```
"unhappiness" → ["un", "##happiness"]
"transformers" → ["transform", "##ers"]  
"HuggingFace" → ["hugging", "##face"]
```

The `##` prefix means "this piece continues the previous token."

### Special Tokens:
- `[CLS]` — placed at the start; its embedding is used for classification
- `[SEP]` — separates sentences (for tasks like question answering)
- `[PAD]` — padding to make all sequences the same length
- `[MASK]` — used during BERT pretraining (masked language model)

In [ ]:
# Tokenization deep dive with BERT's tokenizer

if HF_AVAILABLE:
    try:
        tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
        
        # Basic tokenization
        text = "HuggingFace transformers are amazing!"
        tokens = tokenizer.tokenize(text)
        ids = tokenizer.encode(text)
        
        print("BERT Tokenization")
        print("=" * 60)
        print(f"  Text:    {text}")
        print(f"  Tokens:  {tokens}")
        print(f"  IDs:     {ids}")
        print(f"  Decoded: {tokenizer.decode(ids)}")
        print()
        
        # Show how rare/unusual words get split
        examples = [
            "unhappiness",
            "transformers",
            "antidisestablishmentarianism",
            "ChatGPT is revolutionary",
        ]
        
        print("How BERT handles different words:")
        print("-" * 60)
        for ex in examples:
            toks = tokenizer.tokenize(ex)
            print(f"  '{ex}'")
            print(f"    → {toks}")
            print()
        
        # Full encoding with special tokens and attention mask
        print("Full encoding with special tokens:")
        print("-" * 60)
        encoded = tokenizer(text, padding='max_length', max_length=15, 
                           truncation=True, return_tensors=None)
        
        print(f"  Input IDs:      {encoded['input_ids']}")
        print(f"  Attention Mask:  {encoded['attention_mask']}")
        print(f"  Token Type IDs:  {encoded['token_type_ids']}")
        print()
        
        # Decode to show what each ID means
        decoded_tokens = [tokenizer.decode([tid]) for tid in encoded['input_ids']]
        print("  ID → Token mapping:")
        for tid, tok, mask in zip(encoded['input_ids'], decoded_tokens, encoded['attention_mask']):
            status = "attend" if mask == 1 else "ignore (padding)"
            print(f"    {tid:6d} → '{tok:15s}' [{status}]")
            
    except Exception as e:
        print(f"Tokenizer demo failed: {e}")
else:
    print("Tokenizer demo requires: pip install transformers")
    print()
    print("Example of how BERT tokenizes text:")
    print("  'HuggingFace transformers are amazing!'")
    print("  → ['[CLS]', 'hugging', '##face', 'transformers', 'are', 'amazing', '!', '[SEP]']")
    print("  → [  101,    17662,    12172,      19081,       2024,   6429,    999,  102  ]")

---
## Section 10: Comparing Approaches — Side by Side

Let's put it all together and compare our classical TF-IDF approach with the HuggingFace Transformer on the same movie reviews.

In [ ]:
# Build a simple TF-IDF + LogReg model on the reviews (leave-one-out for small dataset)
from sklearn.model_selection import LeaveOneOut

tfidf_small = TfidfVectorizer()
X_tfidf = tfidf_small.fit_transform(reviews)
y = np.array(labels)

# Leave-one-out cross-validation (necessary for tiny dataset)
loo = LeaveOneOut()
tfidf_preds = np.zeros(len(labels), dtype=int)
for train_idx, test_idx in loo.split(X_tfidf):
    clf = LogisticRegression(max_iter=1000)
    clf.fit(X_tfidf[train_idx], y[train_idx])
    tfidf_preds[test_idx] = clf.predict(X_tfidf[test_idx])

# Build comparison DataFrame
comparison_data = {
    'Review': [r[:50] + '...' for r in reviews],
    'True Label': ['Positive' if l == 1 else 'Negative' for l in labels],
    'TF-IDF + LogReg': ['Positive' if p == 1 else 'Negative' for p in tfidf_preds],
}

# Add HuggingFace predictions if available
if HF_AVAILABLE:
    hf_preds = ['Positive' if r['label'] == 'POSITIVE' else 'Negative' for r in results]
    comparison_data['HuggingFace (BERT)'] = hf_preds

comp_df = pd.DataFrame(comparison_data)

# Mark correct/incorrect
comp_df['TF-IDF Correct'] = comp_df['True Label'] == comp_df['TF-IDF + LogReg']
if HF_AVAILABLE:
    comp_df['HF Correct'] = comp_df['True Label'] == comp_df['HuggingFace (BERT)']

print("Side-by-Side Comparison:")
print("=" * 90)
print(comp_df.to_string(index=False))
print()

# Summary
tfidf_acc = comp_df['TF-IDF Correct'].mean()
print(f"\nTF-IDF + LogReg accuracy: {tfidf_acc:.0%}")
if HF_AVAILABLE:
    hf_acc = comp_df['HF Correct'].mean()
    print(f"HuggingFace BERT accuracy: {hf_acc:.0%}")
    print()
    if hf_acc > tfidf_acc:
        print("The pretrained Transformer outperforms classical NLP on this small dataset!")
        print("It brings knowledge from millions of examples it was trained on.")
    else:
        print("Both approaches perform similarly on this tiny dataset.")
        print("The Transformer advantage grows with more complex/nuanced text.")

In [ ]:
# Visualize the evolution of NLP approaches

fig, ax = plt.subplots(figsize=(14, 6))

approaches = [
    'Bag of Words\n(~2000s)', 
    'TF-IDF\n(~2005)', 
    'Word2Vec\n(2013)', 
    'LSTM/GRU\n(2014-16)', 
    'Transformer\n(2017)', 
    'BERT/GPT\n(2018-19)',
    'GPT-3/4, LLMs\n(2020+)'
]
capabilities = [30, 40, 55, 70, 85, 92, 98]
colors_bar = plt.cm.viridis(np.linspace(0.2, 0.9, len(approaches)))

bars = ax.bar(approaches, capabilities, color=colors_bar, edgecolor='white', linewidth=1.5)

# Add value labels
for bar, val in zip(bars, capabilities):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
            f'{val}%', ha='center', va='bottom', fontweight='bold', fontsize=11)

ax.set_ylabel('Approximate NLP Benchmark Performance', fontsize=12)
ax.set_title('The Evolution of NLP: From Counting Words to Understanding Language', fontsize=14)
ax.set_ylim(0, 110)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Add annotations for key transitions
ax.annotate('Semantic\nunderstanding', xy=(2, 55), xytext=(2, 75),
            arrowprops=dict(arrowstyle='->', color='gray'), ha='center', fontsize=9, color='gray')
ax.annotate('Transfer\nlearning', xy=(5, 92), xytext=(5, 105),
            arrowprops=dict(arrowstyle='->', color='gray'), ha='center', fontsize=9, color='gray')

plt.tight_layout()
save_fig(fig, '05_nlp_evolution')
plt.show()

---
## Key Takeaways

### What We Learned

1. **BoW / TF-IDF** — Simple, fast, interpretable baselines. TF-IDF is almost always better than raw counts. Use these when you need speed, interpretability, or have very limited compute.

2. **Word Embeddings** (Word2Vec, GloVe) — Dense vectors that capture semantic similarity. "good" and "great" become similar, unlike in BoW. The foundation for modern NLP.

3. **RNNs / LSTMs** — Sequence models that process text in order. Important historically, but largely superseded by Transformers due to speed and performance.

4. **Transformers** — The current state of the art. Self-attention lets every word attend to every other word in parallel. BERT (understanding), GPT (generation), and T5 (both) are the key architectures.

5. **HuggingFace** — Makes pretrained models accessible in 3 lines of code. For most tasks, start here.

### The Practitioner's Decision Tree

```
Need NLP for a task?
│
├── Is it a standard task (sentiment, NER, summarization)?
│   └── YES → Use HuggingFace pipeline (2 lines of code)
│
├── Need a custom classifier?
│   ├── Small dataset, need speed → TF-IDF + LogReg
│   └── Large dataset, need accuracy → Fine-tune a pretrained Transformer
│
├── Need interpretability?
│   └── TF-IDF + LogReg (look at feature weights)
│
└── Need text generation?
    └── GPT-based model via HuggingFace or API
```

### What's Next
- **Notebook 06:** We'll explore CNNs for computer vision
- Later: Fine-tuning pretrained models on custom datasets

---
## Exercises

### Exercise 1: TF-IDF + LogReg on 20 Newsgroups
Use TF-IDF + Logistic Regression on the full 20 newsgroups dataset (or pick 4-6 categories of your choice). 
- What accuracy do you get? 
- Which categories are most confused with each other? (Hint: look at the confusion matrix)
- Does increasing `max_features` in TfidfVectorizer help?

In [ ]:
# Exercise 1: Your code here
# Hints:
#   - Try categories like: ['alt.atheism', 'soc.religion.christian', 'sci.med', 'sci.electronics']
#   - These are more similar — harder to classify!
#   - Use sklearn.metrics.confusion_matrix and ConfusionMatrixDisplay

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# TODO: Load data with different categories
# categories = [...]
# train = fetch_20newsgroups(subset='train', categories=categories, remove=('headers', 'footers', 'quotes'))
# test = fetch_20newsgroups(subset='test', categories=categories, remove=('headers', 'footers', 'quotes'))

# TODO: Build and evaluate pipeline
# pipe = Pipeline([...])

# TODO: Plot confusion matrix
# cm = confusion_matrix(y_true, y_pred)
# disp = ConfusionMatrixDisplay(cm, display_labels=train.target_names)
# disp.plot()

### Exercise 2: Compare HuggingFace Sentiment Models
Try different pretrained sentiment models on the same reviews. Do they agree?

Models to try:
- `"distilbert-base-uncased-finetuned-sst-2-english"` (default)
- `"nlptown/bert-base-multilingual-uncased-sentiment"` (1-5 star ratings)
- `"cardiffnlp/twitter-roberta-base-sentiment"` (trained on tweets)

In [ ]:
# Exercise 2: Your code here
# Hints:
#   - pipeline("sentiment-analysis", model="model-name-here")
#   - Some models output different label formats (POSITIVE/NEGATIVE vs 1-5 stars)
#   - Create a DataFrame comparing predictions across models

# TODO: Load multiple models
# model_names = [
#     "distilbert-base-uncased-finetuned-sst-2-english",
#     "nlptown/bert-base-multilingual-uncased-sentiment",
# ]

# TODO: Run each model on the reviews and compare
# for name in model_names:
#     clf = pipeline("sentiment-analysis", model=name)
#     preds = clf(reviews)
#     ...

### Exercise 3: Zero-Shot Classification
Use the `zero-shot-classification` pipeline to classify our movie reviews into custom categories like `["action", "comedy", "drama", "horror"]` — without any training data!

- Does it produce reasonable results?
- What happens if you add more specific labels?
- Try it with non-movie text — does it generalize?

In [ ]:
# Exercise 3: Your code here
# Hints:
#   - zero_shot = pipeline("zero-shot-classification")
#   - result = zero_shot(text, candidate_labels=["action", "comedy", "drama", "horror"])
#   - result['labels'] and result['scores'] give ranked predictions

# TODO: Classify reviews into genre categories
# candidate_labels = ["action", "comedy", "drama", "horror"]
# for review in reviews:
#     result = zero_shot(review, candidate_labels)
#     print(f"Review: {review[:50]}...")
#     print(f"  Top genre: {result['labels'][0]} ({result['scores'][0]:.2f})")

# TODO: Try with different label sets
# candidate_labels_v2 = ["positive review", "negative review", "neutral review"]

# TODO: Try with non-movie text
# news_text = "The Federal Reserve raised interest rates by 0.25% today."
# result = zero_shot(news_text, candidate_labels=["economics", "sports", "technology", "health"])

---

## Summary Table: NLP Methods at a Glance

| Method | Representation | Captures Meaning? | Captures Order? | Speed | When to Use |
|--------|---------------|-------------------|-----------------|-------|------------|
| **Bag of Words** | Sparse counts | No | No | Very fast | Quick baseline, interpretability |
| **TF-IDF** | Sparse weighted | Somewhat (via weighting) | No | Very fast | Better baseline, feature extraction |
| **Word2Vec/GloVe** | Dense vectors | Yes (similarity) | No (per-word) | Fast | Similarity search, clustering |
| **RNN/LSTM** | Hidden states | Yes | Yes (sequential) | Slow | Legacy systems, simple sequences |
| **Transformer** | Contextual embeddings | Yes (deep) | Yes (attention) | Medium | State-of-the-art, most tasks |
| **HuggingFace Pipeline** | Pretrained Transformer | Yes | Yes | Medium | Quick prototyping, production |

---

*Notebook created for Module 2, Week 8 of the Applied AI Engineer learning path.*